# HalluGuard-Qwen3-4B — Version 1-comparable DEV test

This is a **DEV-only engineering test**, not a final benchmark. It matches the Version 1 Bespoke resource envelope where it can be applied without changing HalluGuard's native protocol: Kaggle 2×T4, FP16, and a 4,096-token total context limit. It does not use `MiniCheck.score()` or MiniCheck's 500-token chunking, because both are model-specific and would alter HalluGuard's official document/claim protocol.

The official HalluGuard prompt/XML protocol is retained. Results are not comparable to the Version 1 29,320-example **test** result; this notebook exists only to validate the matching runtime configuration on DEV.

In [ ]:
!pip install -q -U 'transformers>=4.51.0' accelerate datasets pandas scikit-learn

import gc
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'lrsbrgrn/HalluGuard-Qwen3-4B'
DATASET_NAME = 'lytang/LLM-AggreFact'
SPLIT = 'dev'
SMOKE_PER_DATASET_LABEL = 2
# Version 1 Bespoke runtime envelope: Kaggle 2×T4 and max_model_len=4096.
LEGACY_MAX_TOTAL_TOKENS = 4096
LEGACY_DTYPE = torch.float16
OUTPUT_DIR = Path('/kaggle/working/halluguard_v1_comparable_dev')
CHECKPOINT_PATH = OUTPUT_DIR / 'predictions_checkpoint.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SPLIT == 'dev', 'This notebook must remain DEV-only.'
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Run on Kaggle with 2×T4 enabled.'

def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            return token
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

HF_TOKEN = get_hf_token()
print(f'Configured GPUs: {torch.cuda.device_count()}')
print('HF credential source checked; credential values are never printed.')

## DEV sample and native HalluGuard protocol

The sample is reproducible and stratified over the available binary DEV labels. The three HalluGuard categories are preserved independently from the binary mapping.

In [ ]:
dataset_dict = load_dataset(DATASET_NAME, token=HF_TOKEN)
assert 'dev' in dataset_dict and 'test' in dataset_dict
dev_df = pd.DataFrame(dataset_dict['dev']).reset_index(names='source_index')
smoke_df = (
    dev_df.groupby(['dataset', 'label'], group_keys=False)
    .sample(n=SMOKE_PER_DATASET_LABEL, random_state=42)
    .sort_values('source_index').reset_index(drop=True)
)
assert len(smoke_df) > 0 and smoke_df['source_index'].is_unique
print(f'DEV source rows: {len(dev_df):,}; selected smoke rows: {len(smoke_df):,}')
display(smoke_df.groupby(['dataset', 'label']).size().rename('count').reset_index())

VALID_CLASSES = {
    'GROUNDED': 1,
    'HALLUCINATED_INTRINSIC': 0,
    'HALLUCINATED_EXTRINSIC': 0,
}

def create_official_prompt(document, claim):
    return json.dumps({
        'instructions': [
            'You will be given a document and a claim.',
            "Decide whether the claim is 'GROUNDED', 'HALLUCINATED_INTRINSIC', or 'HALLUCINATED_EXTRINSIC' based ONLY on the document.",
            'Definitions:',
            '  - GROUNDED: The claim is fully supported by the document. All relevant parts are directly verifiable from the document.',
            '  - HALLUCINATED_INTRINSIC: The claim contradicts what the document states or clearly implies.',
            '  - HALLUCINATED_EXTRINSIC: The claim includes information that is not stated or implied in the document and cannot be verified using only the document (it requires external knowledge).',
            'Justification requirements:',
            '  - Your justification MUST be evidence-grounded.',
            '  - Explicitly refer to the relevant parts of the document (by quoting or paraphrasing them).',
            '  - Explain how these parts SUPPORT, CONTRADICT, or FAIL TO SUPPORT the claim.',
            '  - Do NOT use any external knowledge; rely only on the provided document.',
            'Answer format (VERY IMPORTANT):',
            '  - You MUST respond using EXACTLY the following XML structure:',
            '    <answer>',
            '      <classification>CATEGORY</classification>',
            '      <justification>Your reasoning here</justification>',
            '    </answer>',
            '  - CATEGORY must be ONE of: GROUNDED, HALLUCINATED_INTRINSIC, HALLUCINATED_EXTRINSIC.',
            '  - The <justification> must briefly explain your reasoning and cite evidence from the document.',
            '  - Do NOT add any other text before or after the <answer>...</answer> block.',
            '  - Do NOT add any extra tags or attributes.',
        ],
        'document': f"'{document}'",
        'claim': f"'{claim}'",
    }, ensure_ascii=False)

def parse_classification(raw_response):
    match = re.search(r'<classification>\s*([^<]+?)\s*</classification>', raw_response, flags=re.IGNORECASE)
    if not match:
        return 'PARSE_ERROR', None
    label = match.group(1).strip().upper()
    return (label, VALID_CLASSES[label]) if label in VALID_CLASSES else ('PARSE_ERROR', None)

for expected_label, expected_binary in VALID_CLASSES.items():
    assert parse_classification(f'<classification>{expected_label}</classification>') == (expected_label, expected_binary)
assert parse_classification('invalid') == ('PARSE_ERROR', None)
print('Native XML parser and all 3-class-to-binary mappings validated.')

## Load with Version 1-compatible resource settings

`LEGACY_MAX_TOTAL_TOKENS=4096` is enforced by shortening only the document before the official prompt is built. This preserves the complete official instruction block and the claim. `document_truncated_for_4096` records every affected row. Generation is capped by the remaining context capacity; it is not tuned from labels.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
max_memory = {gpu_id: '14GiB' for gpu_id in range(torch.cuda.device_count())}
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=LEGACY_DTYPE, device_map='balanced',
    max_memory=max_memory, token=HF_TOKEN
).eval()

# Official sampling values are retained; only the maximum permitted output is constrained by 4096 total tokens.
GENERATION_KWARGS = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'do_sample': True}

def render_prompt(document, claim):
    prompt = create_official_prompt(document, claim)
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}], tokenize=False,
        add_generation_prompt=True, enable_thinking=True
    )

def fit_document_to_legacy_context(document, claim):
    # Binary search for the longest document prefix that leaves at least one generation token.
    document_ids = tokenizer.encode(str(document), add_special_tokens=False)
    low, high, best = 0, len(document_ids), ''
    while low <= high:
        midpoint = (low + high) // 2
        candidate = tokenizer.decode(document_ids[:midpoint], skip_special_tokens=True)
        if len(tokenizer.encode(render_prompt(candidate, claim), add_special_tokens=False)) < LEGACY_MAX_TOTAL_TOKENS:
            best, low = candidate, midpoint + 1
        else:
            high = midpoint - 1
    if not best:
        raise ValueError('Instruction block plus claim exceeds the 4096-token legacy context limit.')
    return best, len(best) < len(str(document))

def generate_halluguard_response(document, claim):
    fitted_document, truncated = fit_document_to_legacy_context(document, claim)
    rendered = render_prompt(fitted_document, claim)
    inputs = tokenizer([rendered], return_tensors='pt').to(model.device)
    available_new_tokens = LEGACY_MAX_TOTAL_TOKENS - inputs.input_ids.shape[1]
    assert available_new_tokens >= 1
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=available_new_tokens, **GENERATION_KWARGS)
    output_ids = generated_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip(), truncated, inputs.input_ids.shape[1], available_new_tokens

def load_checkpoint():
    columns = ['source_index', 'dataset', 'label', 'prediction_3class', 'prediction_binary', 'raw_response', 'status', 'document_truncated_for_4096', 'input_tokens', 'max_new_tokens', 'elapsed_seconds']
    if not CHECKPOINT_PATH.exists():
        return pd.DataFrame(columns=columns)
    frame = pd.read_csv(CHECKPOINT_PATH)
    if frame['source_index'].duplicated().any():
        raise ValueError('Duplicate source_index values in checkpoint.')
    return frame

In [ ]:
results_df = load_checkpoint()
completed = set(results_df['source_index'].astype(int))
remaining_df = smoke_df[~smoke_df['source_index'].isin(completed)]
print(f'Resume state: {len(completed)} saved; {len(remaining_df)} to run.')
for gpu_id in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(gpu_id)

for _, row in remaining_df.iterrows():
    started = time.perf_counter()
    raw_response, truncated, input_tokens, max_new_tokens = '', False, None, None
    try:
        raw_response, truncated, input_tokens, max_new_tokens = generate_halluguard_response(row['doc'], row['claim'])
        prediction_3class, prediction_binary = parse_classification(raw_response)
        status = 'OK' if prediction_3class != 'PARSE_ERROR' else 'PARSE_ERROR'
    except RuntimeError as exc:
        status = 'OOM_ERROR' if 'out of memory' in str(exc).lower() else 'RUNTIME_ERROR'
        prediction_3class, prediction_binary = status, None
        raw_response = f'{type(exc).__name__}: {exc}'
        torch.cuda.empty_cache()
    except Exception as exc:
        status, prediction_3class, prediction_binary = 'RUNTIME_ERROR', 'RUNTIME_ERROR', None
        raw_response = f'{type(exc).__name__}: {exc}'

    saved = {
        'source_index': int(row['source_index']), 'dataset': row['dataset'], 'label': int(row['label']),
        'prediction_3class': prediction_3class, 'prediction_binary': prediction_binary,
        'raw_response': raw_response, 'status': status,
        'document_truncated_for_4096': truncated, 'input_tokens': input_tokens,
        'max_new_tokens': max_new_tokens, 'elapsed_seconds': round(time.perf_counter() - started, 3),
    }
    results_df = pd.concat([results_df, pd.DataFrame([saved])], ignore_index=True)
    results_df.to_csv(CHECKPOINT_PATH, index=False)
    print(f"source_index={saved['source_index']} status={status} input={input_tokens} new_max={max_new_tokens} truncated={truncated}")
    gc.collect(); torch.cuda.empty_cache()

results_df.to_csv(OUTPUT_DIR / 'predictions_dev_v1_comparable_final.csv', index=False)

In [ ]:
results_df = load_checkpoint()
assert set(results_df['source_index'].astype(int)) == set(smoke_df['source_index'].astype(int))
assert results_df['source_index'].is_unique
valid_df = results_df[results_df['status'] == 'OK'].copy()
if len(valid_df):
    valid_df['prediction_binary'] = valid_df['prediction_binary'].astype(int)
    assert set(valid_df['prediction_binary']).issubset({0, 1})
    bacc = balanced_accuracy_score(valid_df['label'], valid_df['prediction_binary']) * 100
else:
    bacc = float('nan')
summary = {
    'stage': 'DEV-only Version 1-resource-comparable smoke test — not final benchmark',
    'model': MODEL_NAME, 'dataset': DATASET_NAME, 'split': SPLIT,
    'examples': int(len(results_df)), 'valid_predictions': int(len(valid_df)),
    'parse_errors': int((results_df['status'] == 'PARSE_ERROR').sum()),
    'all_failures': int((results_df['status'] != 'OK').sum()),
    'document_truncated_for_4096_count': int(results_df['document_truncated_for_4096'].fillna(False).astype(bool).sum()),
    'runtime_seconds_sum_of_examples': round(float(results_df['elapsed_seconds'].sum()), 3),
    'throughput_examples_per_second': round(len(results_df) / float(results_df['elapsed_seconds'].sum()), 4) if float(results_df['elapsed_seconds'].sum()) else None,
    'balanced_accuracy_percent_dev_smoke_only': round(bacc, 2) if pd.notna(bacc) else None,
    'legacy_resource_config': {'gpus': '2x T4', 'dtype': 'float16', 'max_total_tokens': LEGACY_MAX_TOTAL_TOKENS},
    'peak_gpu_memory_gib': {f'gpu_{i}': round(torch.cuda.max_memory_allocated(i) / 1024**3, 3) for i in range(torch.cuda.device_count())},
}
with open(OUTPUT_DIR / 'summary_dev_v1_comparable.json', 'w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)
print(json.dumps(summary, indent=2))
print('This DEV-only score must not be compared directly with Bespoke V1 test BAcc 76.05%.')